# Cadabra 전체 Box trace 직접 합산 검증

이 노트북은 다음 순서를 화면에 그대로 보여줍니다.

1. 첫 코드 셀에서 `main.pdf` 식 (2.1), (2.2), (2.6)의 수치 계수, 부호, tensor body를 executable AST인 `BOX_INPUT`으로 정의합니다.
2. 사용자가 `BOX_INPUT`의 `TensorBodyFactor`, $n$, `(필드, weight)` 8개 조합을 직접 편집할 수 있습니다.
3. 식 (2.3), (2.4)의 total-generator 작용을 대입한 각 필드의 $\Box$를 표시합니다.
4. 각 weighted full trace를 Cadabra 변수 `F1`부터 `F8`까지 저장합니다.
5. 코드 셀에서 실제로 `totalTr = F1 + ... + F8`을 실행합니다.
6. Cadabra의 `collect_terms` 결과가 정확히 `0`인지 보여줍니다.
7. 마지막에는 $\Gamma_{Bq_1q_2}\Gamma^B{}_{q_3q_4}$ body를 $\mathfrak R_{q_1q_2q_3q_4}$로 바꾼 Box를 $T$에만 적용하여 같은 계산 경로가 nonzero를 검출하는지 확인합니다.

기존 Python verifier의 결과 파일이나 CSV를 읽지 않으며, exact rational arithmetic만 사용합니다.

In [1]:
from pathlib import Path
from fractions import Fraction

CORE = Path('full_trace_calculator.cdb').resolve()
exec(compile(CORE.read_text(encoding='utf-8'), str(CORE), 'exec'), globals())

# main.pdf (2.1), (2.2), (2.6)의 실제 editable 계산 입력입니다.
# 계수뿐 아니라 아래 tensor-factor AST를 바꾸면 모든 trace가 다시 계산됩니다.
DELTA_RICCI_BODY_INPUT = BoxTensorBody(
    name='main Ricci',
    monomials=(TensorBodyMonomial(Fraction(1), (
        TensorBodyFactor('Ricci', ('pair1',)),
    )),),
)
DELTA_CONNECTION_BODY_INPUT = BoxTensorBody(
    name='main connection',
    monomials=(TensorBodyMonomial(Fraction(1), (
        TensorBodyFactor('GammaUp', ('coordinate', 'pair1')),
    )),),
)
DELTA_GENERATOR_SQUARE_BODY_INPUT = BoxTensorBody(
    name='main connection square',
    monomials=(TensorBodyMonomial(Fraction(1), (
        TensorBodyFactor('GammaDown', ('coordinate', 'pair1')),
        TensorBodyFactor('GammaUp', ('coordinate', 'pair2')),
    )),),
)
DELTA_MIXED_CURVATURE_BODY_INPUT = BoxTensorBody(
    name='main mixed curvature',
    monomials=(TensorBodyMonomial(Fraction(1), (
        TensorBodyFactor('MixedCurvature', ('left_pair', 'right_pair')),
    )),),
)

# barDelta의 body는 별도 객체이므로 한쪽만 독립적으로 바꿀 수 있습니다.
BAR_DELTA_RICCI_BODY_INPUT = BoxTensorBody(
    name='main Ricci',
    monomials=(TensorBodyMonomial(Fraction(1), (
        TensorBodyFactor('Ricci', ('pair1',)),
    )),),
)
BAR_DELTA_CONNECTION_BODY_INPUT = BoxTensorBody(
    name='main connection',
    monomials=(TensorBodyMonomial(Fraction(1), (
        TensorBodyFactor('GammaUp', ('coordinate', 'pair1')),
    )),),
)
BAR_DELTA_GENERATOR_SQUARE_BODY_INPUT = BoxTensorBody(
    name='main connection square',
    monomials=(TensorBodyMonomial(Fraction(1), (
        TensorBodyFactor('GammaDown', ('coordinate', 'pair1')),
        TensorBodyFactor('GammaUp', ('coordinate', 'pair2')),
    )),),
)
BAR_DELTA_MIXED_CURVATURE_BODY_INPUT = BoxTensorBody(
    name='main mixed curvature',
    monomials=(TensorBodyMonomial(Fraction(1), (
        TensorBodyFactor('MixedCurvature', ('left_pair', 'right_pair')),
    )),),
)

BOX_INPUT = MainBoxDefinition(
    delta=DeltaDefinition(
        laplacian=Fraction(1),
        ricci_generator=Fraction(1),
        connection_derivative_generator=Fraction(-1),
        generator_square=Fraction(1, 4),
        mixed_curvature_generators=Fraction(1, 2),
        ricci_body=DELTA_RICCI_BODY_INPUT,
        connection_body=DELTA_CONNECTION_BODY_INPUT,
        generator_square_body=DELTA_GENERATOR_SQUARE_BODY_INPUT,
        mixed_curvature_body=DELTA_MIXED_CURVATURE_BODY_INPUT,
    ),
    bar_delta=DeltaDefinition(
        laplacian=Fraction(1),
        ricci_generator=Fraction(1),
        connection_derivative_generator=Fraction(-1),
        generator_square=Fraction(1, 4),
        mixed_curvature_generators=Fraction(1, 2),
        ricci_body=BAR_DELTA_RICCI_BODY_INPUT,
        connection_body=BAR_DELTA_CONNECTION_BODY_INPUT,
        generator_square_body=BAR_DELTA_GENERATOR_SQUARE_BODY_INPUT,
        mixed_curvature_body=BAR_DELTA_MIXED_CURVATURE_BODY_INPUT,
    ),
    delta_in_box=Fraction(1),
    bar_delta_in_box=Fraction(-1),
)

print('Loaded:', CORE.name)
print('Editable main.pdf input:', BOX_INPUT)
print('Matches main.pdf defaults:', BOX_INPUT == MAIN_PDF_BOX)

Loaded: full_trace_calculator.cdb
Editable main.pdf input: MainBoxDefinition(delta=DeltaDefinition(laplacian=Fraction(1, 1), ricci_generator=Fraction(1, 1), connection_derivative_generator=Fraction(-1, 1), generator_square=Fraction(1, 4), mixed_curvature_generators=Fraction(1, 2), ricci_body=BoxTensorBody(name='main Ricci', monomials=(TensorBodyMonomial(coefficient=Fraction(1, 1), factors=(TensorBodyFactor(kind='Ricci', slots=('pair1',)),)),)), connection_body=BoxTensorBody(name='main connection', monomials=(TensorBodyMonomial(coefficient=Fraction(1, 1), factors=(TensorBodyFactor(kind='GammaUp', slots=('coordinate', 'pair1')),)),)), generator_square_body=BoxTensorBody(name='main connection square', monomials=(TensorBodyMonomial(coefficient=Fraction(1, 1), factors=(TensorBodyFactor(kind='GammaDown', slots=('coordinate', 'pair1')), TensorBodyFactor(kind='GammaUp', slots=('coordinate', 'pair2')))),)), mixed_curvature_body=BoxTensorBody(name='main mixed curvature', monomials=(TensorBodyMon

In [2]:
print('Supported n: 1, 2')
print('Box =', BOX_INPUT.delta_in_box, '* Delta +', BOX_INPUT.bar_delta_in_box, '* barDelta')
print('Delta coefficients    =', BOX_INPUT.delta)
print('barDelta coefficients =', BOX_INPUT.bar_delta)

Supported n: 1, 2
Box = 1 * Delta + -1 * barDelta
Delta coefficients    = DeltaDefinition(laplacian=Fraction(1, 1), ricci_generator=Fraction(1, 1), connection_derivative_generator=Fraction(-1, 1), generator_square=Fraction(1, 4), mixed_curvature_generators=Fraction(1, 2), ricci_body=BoxTensorBody(name='main Ricci', monomials=(TensorBodyMonomial(coefficient=Fraction(1, 1), factors=(TensorBodyFactor(kind='Ricci', slots=('pair1',)),)),)), connection_body=BoxTensorBody(name='main connection', monomials=(TensorBodyMonomial(coefficient=Fraction(1, 1), factors=(TensorBodyFactor(kind='GammaUp', slots=('coordinate', 'pair1')),)),)), generator_square_body=BoxTensorBody(name='main connection square', monomials=(TensorBodyMonomial(coefficient=Fraction(1, 1), factors=(TensorBodyFactor(kind='GammaDown', slots=('coordinate', 'pair1')), TensorBodyFactor(kind='GammaUp', slots=('coordinate', 'pair2')))),)), mixed_curvature_body=BoxTensorBody(name='main mixed curvature', monomials=(TensorBodyMonomial(coe

## 1. 입력한 `main.pdf`의 $\Box$ 정의를 표시

첫 코드 셀의 `BOX_INPUT`이 아래 식의 계수와 부호를 결정합니다. 기본값은 `main.pdf` 식 (2.1), (2.2), (2.6)과 정확히 같습니다. $\mathcal A_X$, $\mathcal B_X$, $h$, $u$, $b$ 같은 재인수분해 기호를 도입하지 않습니다. $n=2$에서는 같은 문서의 식 (2.5)에 따라 오른쪽 total generator가 먼저 작용하는 순서를 사용합니다.

In [3]:
BOX_DEFINITION = show_box_definition(box_definition=BOX_INPUT)

${}\begin{aligned}\Box T:&=\Delta T\\&\quad{}-\bar\Delta T.\end{aligned}\tag{2.6}$

${}\begin{aligned}\Delta T&=\mathcal D_q\mathcal D^qT\\&\quad{}+\mathfrak R_{[q_1q_2]}G^{q_1q_2}T\\&\quad{}-\Gamma^B{}_{q_1q_2}\mathcal D_BG^{q_1q_2}T\\&\quad{}+\frac{1}{4}\,\Gamma_{Bq_1q_2}\Gamma^B{}_{q_3q_4}G^{q_1q_2}G^{q_3q_4}T\\&\quad{}+\frac{1}{2}\,\mathfrak R_{\bar q_1\bar q_2q_3q_4}\bar G^{\bar q_1\bar q_2}G^{q_3q_4}T.\end{aligned}\tag{2.1}$

${}\begin{aligned}\bar\Delta T&=\mathcal D_{\bar q}\mathcal D^{\bar q}T\\&\quad{}+\mathfrak R_{[\bar q_1\bar q_2]}\bar G^{\bar q_1\bar q_2}T\\&\quad{}-\Gamma^B{}_{\bar q_1\bar q_2}\mathcal D_B\bar G^{\bar q_1\bar q_2}T\\&\quad{}+\frac{1}{4}\,\Gamma_{B\bar q_1\bar q_2}\Gamma^B{}_{\bar q_3\bar q_4}\bar G^{\bar q_1\bar q_2}\bar G^{\bar q_3\bar q_4}T\\&\quad{}+\frac{1}{2}\,\mathfrak R_{q_1q_2\bar q_3\bar q_4}G^{q_1q_2}\bar G^{\bar q_3\bar q_4}T.\end{aligned}\tag{2.2}$

## 2. $n$과 필드 조합 입력

각 tuple의 두 번째 값은 그 필드 전체 trace 하나에 곱하는 weight입니다. 개별 tensor 항마다 다른 weight를 주는 것이 아닙니다. 기본값은 기존 8필드 $n=2$ 조합입니다.

In [4]:
N = 2

FIELD_COMBINATION = (
    ('T',    Fraction(1)),
    ('phi',  Fraction(128)),
    ('BLL',  Fraction(1, 4)),
    ('BRR',  Fraction(1, 4)),
    ('UL',   Fraction(-12)),
    ('UR',   Fraction(-12)),
    ('ULLR', Fraction(-1, 64)),
    ('ULRR', Fraction(-1, 64)),
)

# 정상 검증에서는 여덟 필드가 정확히 같은 main.pdf Box 입력을 공유합니다.
FIELD_BOX_DEFINITIONS = {
    field: BOX_INPUT for field, _weight in FIELD_COMBINATION
}

assert N in (1, 2)
assert len(FIELD_COMBINATION) == 8
print('n =', N)
for index, (field, weight) in enumerate(FIELD_COMBINATION, 1):
    print(f'F{index}: field={field:5s}, weight={weight}')

n = 2
F1: field=T    , weight=1
F2: field=phi  , weight=128
F3: field=BLL  , weight=1/4
F4: field=BRR  , weight=1/4
F5: field=UL   , weight=-12
F6: field=UR   , weight=-12
F7: field=ULLR , weight=-1/64
F8: field=ULRR , weight=-1/64


## 3. 각 필드에 작용하는 $\Box$

각 식은 `main.pdf` 식 (2.1)–(2.6)을 해당 필드 성분에 직접 특수화한 것입니다. 식 (2.3), (2.4)의 $G$, $\bar G$ 작용도 실제 $\alpha_i$, $\bar\beta_j$ 지수로 항별로 표시하며 슬롯 약식기호를 쓰지 않습니다. $a=0$ 또는 $b=0$인 sector의 generator 항은 직접 사라집니다.

In [5]:
FIELD_BOXES = show_field_boxes(
    FIELD_COMBINATION,
    box_definition=BOX_INPUT,
    field_box_definitions=FIELD_BOX_DEFINITIONS,
)
assert len(FIELD_BOXES) == 8

${}\begin{aligned}T\in S^{\otimes 1}\otimes(\bar S^*)^{\otimes 1},\\\left(G^{q_1q_2}T\right)^{\alpha}{}_{\bar\beta}=\frac12(\gamma^{q_1q_2})^{\alpha}{}_{\rho}T^{\rho}{}_{\bar\beta},\qquad \left(\bar G^{\bar q_1\bar q_2}T\right)^{\alpha}{}_{\bar\beta}=-\frac12T^{\alpha}{}_{\bar\rho}(\bar\gamma^{\bar q_1\bar q_2})^{\bar\rho}{}_{\bar\beta},\\\Box T&=\mathcal D_q\mathcal D^qT\\&\quad{}+\mathfrak R_{[q_1q_2]}G^{q_1q_2}T\\&\quad{}-\Gamma^B{}_{q_1q_2}\mathcal D_BG^{q_1q_2}T\\&\quad{}+\frac{1}{4}\,\Gamma_{Bq_1q_2}\Gamma^B{}_{q_3q_4}G^{q_1q_2}G^{q_3q_4}T\\&\quad{}+\frac{1}{2}\,\mathfrak R_{\bar q_1\bar q_2q_3q_4}\bar G^{\bar q_1\bar q_2}G^{q_3q_4}T\\&\quad{}-\mathcal D_{\bar q}\mathcal D^{\bar q}T\\&\quad{}-\mathfrak R_{[\bar q_1\bar q_2]}\bar G^{\bar q_1\bar q_2}T\\&\quad{}+\Gamma^B{}_{\bar q_1\bar q_2}\mathcal D_B\bar G^{\bar q_1\bar q_2}T\\&\quad{}-\frac{1}{4}\,\Gamma_{B\bar q_1\bar q_2}\Gamma^B{}_{\bar q_3\bar q_4}\bar G^{\bar q_1\bar q_2}\bar G^{\bar q_3\bar q_4}T\\&\quad{}-\frac{1}{2}\,\mathfrak R_{q_1q_2\bar q_3\bar q_4}G^{q_1q_2}\bar G^{\bar q_3\bar q_4}T.\end{aligned}$

${}\begin{aligned}\phi\in S^{\otimes 0}\otimes(\bar S^*)^{\otimes 0},\\\left(G^{q_1q_2}\phi\right)=0,\qquad \left(\bar G^{\bar q_1\bar q_2}\phi\right)=0,\\\Box \phi&=\mathcal D_q\mathcal D^q\phi\\&\quad{}-\mathcal D_{\bar q}\mathcal D^{\bar q}\phi.\end{aligned}$

${}\begin{aligned}B_{LL}\in S^{\otimes 2}\otimes(\bar S^*)^{\otimes 0},\\\left(G^{q_1q_2}B_{LL}\right)^{\alpha_1\alpha_2}=\frac12(\gamma^{q_1q_2})^{\alpha_1}{}_{\rho}B_{LL}^{\rho\alpha_2}+\frac12(\gamma^{q_1q_2})^{\alpha_2}{}_{\rho}B_{LL}^{\alpha_1\rho},\qquad \left(\bar G^{\bar q_1\bar q_2}B_{LL}\right)=0,\\\Box B_{LL}&=\mathcal D_q\mathcal D^qB_{LL}\\&\quad{}+\mathfrak R_{[q_1q_2]}G^{q_1q_2}B_{LL}\\&\quad{}-\Gamma^B{}_{q_1q_2}\mathcal D_BG^{q_1q_2}B_{LL}\\&\quad{}+\frac{1}{4}\,\Gamma_{Bq_1q_2}\Gamma^B{}_{q_3q_4}G^{q_1q_2}G^{q_3q_4}B_{LL}\\&\quad{}-\mathcal D_{\bar q}\mathcal D^{\bar q}B_{LL}.\end{aligned}$

${}\begin{aligned}B_{RR}\in S^{\otimes 0}\otimes(\bar S^*)^{\otimes 2},\\\left(G^{q_1q_2}B_{RR}\right)=0,\qquad \left(\bar G^{\bar q_1\bar q_2}B_{RR}\right){}_{\bar\beta_1\bar\beta_2}=-\frac12B_{RR}{}_{\bar\rho\bar\beta_2}(\bar\gamma^{\bar q_1\bar q_2})^{\bar\rho}{}_{\bar\beta_1}-\frac12B_{RR}{}_{\bar\beta_1\bar\rho}(\bar\gamma^{\bar q_1\bar q_2})^{\bar\rho}{}_{\bar\beta_2},\\\Box B_{RR}&=\mathcal D_q\mathcal D^qB_{RR}\\&\quad{}-\mathcal D_{\bar q}\mathcal D^{\bar q}B_{RR}\\&\quad{}-\mathfrak R_{[\bar q_1\bar q_2]}\bar G^{\bar q_1\bar q_2}B_{RR}\\&\quad{}+\Gamma^B{}_{\bar q_1\bar q_2}\mathcal D_B\bar G^{\bar q_1\bar q_2}B_{RR}\\&\quad{}-\frac{1}{4}\,\Gamma_{B\bar q_1\bar q_2}\Gamma^B{}_{\bar q_3\bar q_4}\bar G^{\bar q_1\bar q_2}\bar G^{\bar q_3\bar q_4}B_{RR}.\end{aligned}$

${}\begin{aligned}U_L\in S^{\otimes 1}\otimes(\bar S^*)^{\otimes 0},\\\left(G^{q_1q_2}U_L\right)^{\alpha}=\frac12(\gamma^{q_1q_2})^{\alpha}{}_{\rho}U_L^{\rho},\qquad \left(\bar G^{\bar q_1\bar q_2}U_L\right)=0,\\\Box U_L&=\mathcal D_q\mathcal D^qU_L\\&\quad{}+\mathfrak R_{[q_1q_2]}G^{q_1q_2}U_L\\&\quad{}-\Gamma^B{}_{q_1q_2}\mathcal D_BG^{q_1q_2}U_L\\&\quad{}+\frac{1}{4}\,\Gamma_{Bq_1q_2}\Gamma^B{}_{q_3q_4}G^{q_1q_2}G^{q_3q_4}U_L\\&\quad{}-\mathcal D_{\bar q}\mathcal D^{\bar q}U_L.\end{aligned}$

${}\begin{aligned}U_R\in S^{\otimes 0}\otimes(\bar S^*)^{\otimes 1},\\\left(G^{q_1q_2}U_R\right)=0,\qquad \left(\bar G^{\bar q_1\bar q_2}U_R\right){}_{\bar\beta}=-\frac12U_{R\,\bar\rho}(\bar\gamma^{\bar q_1\bar q_2})^{\bar\rho}{}_{\bar\beta},\\\Box U_R&=\mathcal D_q\mathcal D^qU_R\\&\quad{}-\mathcal D_{\bar q}\mathcal D^{\bar q}U_R\\&\quad{}-\mathfrak R_{[\bar q_1\bar q_2]}\bar G^{\bar q_1\bar q_2}U_R\\&\quad{}+\Gamma^B{}_{\bar q_1\bar q_2}\mathcal D_B\bar G^{\bar q_1\bar q_2}U_R\\&\quad{}-\frac{1}{4}\,\Gamma_{B\bar q_1\bar q_2}\Gamma^B{}_{\bar q_3\bar q_4}\bar G^{\bar q_1\bar q_2}\bar G^{\bar q_3\bar q_4}U_R.\end{aligned}$

${}\begin{aligned}U_{LLR}\in S^{\otimes 2}\otimes(\bar S^*)^{\otimes 1},\\\left(G^{q_1q_2}U_{LLR}\right)^{\alpha_1\alpha_2}{}_{\bar\beta}=\frac12(\gamma^{q_1q_2})^{\alpha_1}{}_{\rho}U_{LLR}^{\rho\alpha_2}{}_{\bar\beta}+\frac12(\gamma^{q_1q_2})^{\alpha_2}{}_{\rho}U_{LLR}^{\alpha_1\rho}{}_{\bar\beta},\qquad \left(\bar G^{\bar q_1\bar q_2}U_{LLR}\right)^{\alpha_1\alpha_2}{}_{\bar\beta}=-\frac12U_{LLR}^{\alpha_1\alpha_2}{}_{\bar\rho}(\bar\gamma^{\bar q_1\bar q_2})^{\bar\rho}{}_{\bar\beta},\\\Box U_{LLR}&=\mathcal D_q\mathcal D^qU_{LLR}\\&\quad{}+\mathfrak R_{[q_1q_2]}G^{q_1q_2}U_{LLR}\\&\quad{}-\Gamma^B{}_{q_1q_2}\mathcal D_BG^{q_1q_2}U_{LLR}\\&\quad{}+\frac{1}{4}\,\Gamma_{Bq_1q_2}\Gamma^B{}_{q_3q_4}G^{q_1q_2}G^{q_3q_4}U_{LLR}\\&\quad{}+\frac{1}{2}\,\mathfrak R_{\bar q_1\bar q_2q_3q_4}\bar G^{\bar q_1\bar q_2}G^{q_3q_4}U_{LLR}\\&\quad{}-\mathcal D_{\bar q}\mathcal D^{\bar q}U_{LLR}\\&\quad{}-\mathfrak R_{[\bar q_1\bar q_2]}\bar G^{\bar q_1\bar q_2}U_{LLR}\\&\quad{}+\Gamma^B{}_{\bar q_1\bar q_2}\mathcal D_B\bar G^{\bar q_1\bar q_2}U_{LLR}\\&\quad{}-\frac{1}{4}\,\Gamma_{B\bar q_1\bar q_2}\Gamma^B{}_{\bar q_3\bar q_4}\bar G^{\bar q_1\bar q_2}\bar G^{\bar q_3\bar q_4}U_{LLR}\\&\quad{}-\frac{1}{2}\,\mathfrak R_{q_1q_2\bar q_3\bar q_4}G^{q_1q_2}\bar G^{\bar q_3\bar q_4}U_{LLR}.\end{aligned}$

${}\begin{aligned}U_{LRR}\in S^{\otimes 1}\otimes(\bar S^*)^{\otimes 2},\\\left(G^{q_1q_2}U_{LRR}\right)^{\alpha}{}_{\bar\beta_1\bar\beta_2}=\frac12(\gamma^{q_1q_2})^{\alpha}{}_{\rho}U_{LRR}^{\rho}{}_{\bar\beta_1\bar\beta_2},\qquad \left(\bar G^{\bar q_1\bar q_2}U_{LRR}\right)^{\alpha}{}_{\bar\beta_1\bar\beta_2}=-\frac12U_{LRR}^{\alpha}{}_{\bar\rho\bar\beta_2}(\bar\gamma^{\bar q_1\bar q_2})^{\bar\rho}{}_{\bar\beta_1}-\frac12U_{LRR}^{\alpha}{}_{\bar\beta_1\bar\rho}(\bar\gamma^{\bar q_1\bar q_2})^{\bar\rho}{}_{\bar\beta_2},\\\Box U_{LRR}&=\mathcal D_q\mathcal D^qU_{LRR}\\&\quad{}+\mathfrak R_{[q_1q_2]}G^{q_1q_2}U_{LRR}\\&\quad{}-\Gamma^B{}_{q_1q_2}\mathcal D_BG^{q_1q_2}U_{LRR}\\&\quad{}+\frac{1}{4}\,\Gamma_{Bq_1q_2}\Gamma^B{}_{q_3q_4}G^{q_1q_2}G^{q_3q_4}U_{LRR}\\&\quad{}+\frac{1}{2}\,\mathfrak R_{\bar q_1\bar q_2q_3q_4}\bar G^{\bar q_1\bar q_2}G^{q_3q_4}U_{LRR}\\&\quad{}-\mathcal D_{\bar q}\mathcal D^{\bar q}U_{LRR}\\&\quad{}-\mathfrak R_{[\bar q_1\bar q_2]}\bar G^{\bar q_1\bar q_2}U_{LRR}\\&\quad{}+\Gamma^B{}_{\bar q_1\bar q_2}\mathcal D_B\bar G^{\bar q_1\bar q_2}U_{LRR}\\&\quad{}-\frac{1}{4}\,\Gamma_{B\bar q_1\bar q_2}\Gamma^B{}_{\bar q_3\bar q_4}\bar G^{\bar q_1\bar q_2}\bar G^{\bar q_3\bar q_4}U_{LRR}\\&\quad{}-\frac{1}{2}\,\mathfrak R_{q_1q_2\bar q_3\bar q_4}G^{q_1q_2}\bar G^{\bar q_3\bar q_4}U_{LRR}.\end{aligned}$

## 4. 각 weighted full trace를 `F1`–`F8`에 저장

`prepare_field_variables`는 각 필드의 $\operatorname{tr}_X(\Box_X^n)$를 끝까지 전개하고, 입력한 field weight를 전체 식에 곱합니다. 아래 셀은 그 8개 완전 전개식을 모두 렌더링한 뒤 실제 Cadabra `Ex` 객체를 `F1`–`F8`에 저장합니다. 이 `Ex`들은 `K0001` 같은 대리 기호가 아니라 $H,\Gamma,\Phi,\mathfrak R$ 텐서와 실제 partial derivative를 담습니다. 오른쪽 작용 외부 미분은 공통의 임의 시험함수 `Probe`에 작용시켜 완전 수축된 스칼라 연산자로 표현합니다.

In [6]:
PREPARED = prepare_field_variables(
    FIELD_COMBINATION,
    N,
    box_definition=BOX_INPUT,
    field_box_definitions=FIELD_BOX_DEFINITIONS,
    show=True,
    terms_per_chunk=15,
)

F1, F2, F3, F4, F5, F6, F7, F8 = PREPARED.weighted_expressions

assert PREPARED.variable_names == ('F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8')
cadabra_payloads = tuple(
    expression._latex_()
    for expression in (F1, F2, F3, F4, F5, F6, F7, F8)
)
assert all(
    symbol not in payload
    for payload in cadabra_payloads
    for symbol in PREPARED.symbols.values()
)
print('stored variables:', ', '.join(PREPARED.variable_names))
print('Cadabra payload: actual tensor/partial expressions; K-labels: 0')
print('all expanded rows:', PREPARED.term_count)

${}\begin{aligned}F1\equiv\left(1\right)\operatorname{tr}_{T}\!\left(\Box\circ\Box\right)&=256\,\mathcal H^{AB}\,\mathcal H^{CD}\,\partial_{A}\,\partial_{B}\,\partial_{C}\,\partial_{D}\\&\quad{}+512\,\Gamma_{MN}{}^{C}\,\mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{A}\,\partial_{B}\,\partial_{C}\\&\quad{}+512\,\mathcal H^{AB}\,\partial_{A}\mathcal H^{CD}\,\partial_{B}\,\partial_{C}\,\partial_{D}\\&\quad{}-64\,\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{A}\,\partial_{B}\\&\quad{}-128\,\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{CN}\,\mathcal H^{EM}\,\partial_{E}\,\partial_{C}\\&\quad{}-128\,\bar\Phi_{M\bar p\bar q}\,\Gamma^{C\bar p\bar q}\,\mathcal H^{EM}\,\partial_{E}\,\partial_{C}\\&\quad{}-128\,\bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma^{E}{}_{\bar p\bar q}\,\mathcal H^{CM}\,\partial_{E}\,\partial_{C}\\&\quad{}-128\,\bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma^{M}{}_{\bar p\bar q}\,\mathcal H^{AB}\,\partial_{A}\,\partial_{B}\\&\quad{}+512\,\partial_{A}\Gamma_{MN}{}^{C}\,\mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{B}\,\partial_{C}\\&\quad{}+512\,\Gamma_{MN}{}^{C}\,\mathcal H^{AB}\,\partial_{A}\mathcal H^{MN}\,\partial_{B}\,\partial_{C}\\&\quad{}+256\,\Gamma_{MN}{}^{E}\,\Gamma_{PQ}{}^{C}\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\partial_{E}\,\partial_{C}\\&\quad{}+256\,\Gamma_{MN}{}^{E}\,\partial_{E}\mathcal H^{CD}\,\mathcal H^{MN}\,\partial_{C}\,\partial_{D}\\&\quad{}-64\,\Gamma_{Mpq}\,\Gamma^{Mpq}\,\mathcal H^{AB}\,\partial_{A}\,\partial_{B}\\&\quad{}-128\,\Gamma^{Cpq}\,\Gamma^{E}{}_{pq}\,\partial_{E}\,\partial_{C}\\&\quad{}+128\,\Gamma^{Cpq}\,\mathcal H^{EM}\,\Phi_{Mpq}\,\partial_{E}\,\partial_{C}\end{aligned}$

${}\begin{aligned}&\quad{}+128\,\Gamma^{E}{}_{pq}\,\mathcal H^{CM}\,\Phi_{M}{}^{pq}\,\partial_{E}\,\partial_{C}\\&\quad{}+128\,\Gamma^{M}{}_{pq}\,\mathcal H^{AB}\,\Phi_{M}{}^{pq}\,\partial_{A}\,\partial_{B}\\&\quad{}+64\,\Gamma_{M\bar p\bar q}\,\Gamma^{M\bar p\bar q}\,\mathcal H^{AB}\,\partial_{A}\,\partial_{B}\\&\quad{}-128\,\Gamma^{C\bar p\bar q}\,\Gamma^{E}{}_{\bar p\bar q}\,\partial_{E}\,\partial_{C}\\&\quad{}+256\,\mathcal H^{AB}\,\partial_{A}\partial_{B}\mathcal H^{CD}\,\partial_{C}\,\partial_{D}\\&\quad{}-64\,\mathcal H^{AB}\,\mathcal H^{MN}\,\Phi_{Mpq}\,\Phi_{N}{}^{pq}\,\partial_{A}\,\partial_{B}\\&\quad{}-128\,\mathcal H^{CM}\,\mathcal H^{EN}\,\Phi_{M}{}^{pq}\,\Phi_{Npq}\,\partial_{E}\,\partial_{C}\\&\quad{}-64\,\partial_{A}\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{AC}\,\mathcal H^{MN}\,\partial_{C}\\&\quad{}-64\,\partial_{N}\bar\Phi_{M\bar p\bar q}\,\Gamma^{C\bar p\bar q}\,\mathcal H^{NM}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M\bar p\bar q}\,\partial_{A}\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{AC}\,\mathcal H^{MN}\,\partial_{C}\\&\quad{}-128\,\bar\Phi_{M\bar p\bar q}\,\partial_{E}\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{CN}\,\mathcal H^{EM}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M\bar p\bar q}\,\partial_{P}\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{CM}\,\mathcal H^{PN}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\Gamma_{PQ}{}^{C}\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\Gamma_{PQ}{}^{M}\,\mathcal H^{CN}\,\mathcal H^{PQ}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{AC}\,\partial_{A}\mathcal H^{MN}\,\partial_{C}\end{aligned}$

${}\begin{aligned}&\quad{}-128\,\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\partial_{E}\mathcal H^{CN}\,\mathcal H^{EM}\,\partial_{C}\\&\quad{}+64\,\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p}{}_{\bar r}\,\bar\Phi_{P}{}^{\bar q\bar r}\,\mathcal H^{CM}\,\mathcal H^{NP}\,\partial_{C}\\&\quad{}+64\,\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p}{}_{\bar r}\,\Gamma^{C\bar q\bar r}\,\mathcal H^{MN}\,\partial_{C}\\&\quad{}+64\,\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar q\bar r}\,\bar\Phi_{P}{}^{\bar p}{}_{\bar r}\,\mathcal H^{CN}\,\mathcal H^{MP}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M\bar p\bar q}\,\Gamma_{NP}{}^{M}\,\Gamma^{C\bar p\bar q}\,\mathcal H^{NP}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M\bar p\bar q}\,\Gamma_{N}{}^{\bar p}{}_{\bar r}\,\Gamma^{N\bar q\bar r}\,\mathcal H^{CM}\,\partial_{C}\\&\quad{}-128\,\bar\Phi_{M\bar p\bar q}\,\partial_{E}\Gamma^{C\bar p\bar q}\,\mathcal H^{EM}\,\partial_{C}\\&\quad{}+128\,\bar\Phi_{M\bar p\bar q}\,\mathcal H^{CM}\,\mathfrak R^{[\bar p\bar q]}\,\partial_{C}\\&\quad{}-128\,\partial_{A}\bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma^{M}{}_{\bar p\bar q}\,\mathcal H^{AC}\,\partial_{C}\\&\quad{}-128\,\partial_{E}\bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma^{E}{}_{\bar p\bar q}\,\mathcal H^{CM}\,\partial_{C}\\&\quad{}-64\,\partial_{N}\bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma^{C}{}_{\bar p\bar q}\,\mathcal H^{NM}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M}{}^{\bar p\bar q}\,\partial_{P}\bar\Phi_{N\bar p\bar q}\,\mathcal H^{CM}\,\mathcal H^{PN}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M}{}^{\bar p\bar q}\,\bar\Phi_{N\bar p\bar q}\,\Gamma_{PQ}{}^{M}\,\mathcal H^{CN}\,\mathcal H^{PQ}\,\partial_{C}\\&\quad{}-128\,\bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma_{NP}{}^{C}\,\Gamma^{M}{}_{\bar p\bar q}\,\mathcal H^{NP}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma_{NP}{}^{M}\,\Gamma^{C}{}_{\bar p\bar q}\,\mathcal H^{NP}\,\partial_{C}\end{aligned}$

${}\begin{aligned}&\quad{}-128\,\bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma^{E}{}_{\bar p\bar q}\,\partial_{E}\mathcal H^{CM}\,\partial_{C}\\&\quad{}-128\,\bar\Phi_{M}{}^{\bar p\bar q}\,\partial_{A}\Gamma^{M}{}_{\bar p\bar q}\,\mathcal H^{AC}\,\partial_{C}\\&\quad{}+128\,\bar\Phi_{M}{}^{\bar p\bar q}\,\mathcal H^{CM}\,\mathfrak R_{[\bar p\bar q]}\,\partial_{C}\\&\quad{}+64\,\bar\Phi_{M}{}^{\bar p}{}_{\bar r}\,\bar\Phi_{N}{}^{\bar q\bar r}\,\Gamma^{C}{}_{\bar p\bar q}\,\mathcal H^{MN}\,\partial_{C}\\&\quad{}+128\,\bar\Phi_{M}{}^{\bar p}{}_{\bar r}\,\bar\Phi_{N}{}^{\bar q\bar r}\,\Gamma^{M}{}_{\bar p\bar q}\,\mathcal H^{CN}\,\partial_{C}\\&\quad{}+128\,\bar\Phi_{M}{}^{\bar p}{}_{\bar r}\,\Gamma^{C\bar q\bar r}\,\Gamma^{M}{}_{\bar p\bar q}\,\partial_{C}\\&\quad{}+128\,\bar\Phi_{M}{}^{\bar q\bar r}\,\bar\Phi_{N\bar p\bar q}\,\Gamma^{M\bar p}{}_{\bar r}\,\mathcal H^{CN}\,\partial_{C}\\&\quad{}-64\,\bar\Phi_{M}{}^{\bar q\bar r}\,\Gamma_{N\bar p\bar q}\,\Gamma^{N\bar p}{}_{\bar r}\,\mathcal H^{CM}\,\partial_{C}\\&\quad{}+128\,\bar\Phi_{M}{}^{\bar q\bar r}\,\Gamma^{C}{}_{\bar p\bar q}\,\Gamma^{M\bar p}{}_{\bar r}\,\partial_{C}\\&\quad{}+256\,\partial_{A}\partial_{B}\Gamma_{MN}{}^{C}\,\mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{C}\\&\quad{}+256\,\partial_{A}\Gamma_{MN}{}^{C}\,\mathcal H^{AB}\,\partial_{B}\mathcal H^{MN}\,\partial_{C}\\&\quad{}+256\,\partial_{B}\Gamma_{MN}{}^{C}\,\mathcal H^{AB}\,\partial_{A}\mathcal H^{MN}\,\partial_{C}\\&\quad{}-64\,\Gamma_{MN}{}^{C}\,\Gamma_{Ppq}\,\Gamma^{Ppq}\,\mathcal H^{MN}\,\partial_{C}\\&\quad{}+128\,\Gamma_{MN}{}^{C}\,\Gamma^{P}{}_{pq}\,\mathcal H^{MN}\,\Phi_{P}{}^{pq}\,\partial_{C}\\&\quad{}+64\,\Gamma_{MN}{}^{C}\,\Gamma_{P\bar p\bar q}\,\Gamma^{P\bar p\bar q}\,\mathcal H^{MN}\,\partial_{C}\end{aligned}$

${}\begin{aligned}&\quad{}+256\,\Gamma_{MN}{}^{C}\,\mathcal H^{AB}\,\partial_{A}\partial_{B}\mathcal H^{MN}\,\partial_{C}\\&\quad{}-64\,\Gamma_{MN}{}^{C}\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Ppq}\,\Phi_{Q}{}^{pq}\,\partial_{C}\\&\quad{}+256\,\Gamma_{MN}{}^{E}\,\partial_{E}\Gamma_{PQ}{}^{C}\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\partial_{C}\\&\quad{}+256\,\Gamma_{MN}{}^{E}\,\Gamma_{PQ}{}^{C}\,\mathcal H^{MN}\,\partial_{E}\mathcal H^{PQ}\,\partial_{C}\\&\quad{}+64\,\Gamma_{MN}{}^{P}\,\Gamma^{C}{}_{pq}\,\mathcal H^{MN}\,\Phi_{P}{}^{pq}\,\partial_{C}\\&\quad{}+64\,\Gamma_{MN}{}^{P}\,\Gamma^{Cpq}\,\mathcal H^{MN}\,\Phi_{Ppq}\,\partial_{C}\\&\quad{}-64\,\Gamma_{MN}{}^{P}\,\mathcal H^{CQ}\,\mathcal H^{MN}\,\Phi_{Ppq}\,\Phi_{Q}{}^{pq}\,\partial_{C}\\&\quad{}-64\,\Gamma_{MN}{}^{P}\,\mathcal H^{CQ}\,\mathcal H^{MN}\,\Phi_{P}{}^{pq}\,\Phi_{Qpq}\,\partial_{C}\\&\quad{}-64\,\partial_{A}\Gamma_{Mpq}\,\Gamma^{Mpq}\,\mathcal H^{AC}\,\partial_{C}\\&\quad{}-64\,\Gamma_{Mpq}\,\Gamma^{Cqr}\,\Gamma^{Mp}{}_{r}\,\partial_{C}\\&\quad{}-64\,\Gamma_{Mpq}\,\partial_{A}\Gamma^{Mpq}\,\mathcal H^{AC}\,\partial_{C}\\&\quad{}+64\,\Gamma_{Mpq}\,\Gamma^{Mp}{}_{r}\,\mathcal H^{CN}\,\Phi_{N}{}^{qr}\,\partial_{C}\\&\quad{}-64\,\Gamma_{M}{}^{p}{}_{r}\,\Gamma^{C}{}_{pq}\,\Gamma^{Mqr}\,\partial_{C}\\&\quad{}+64\,\Gamma_{M}{}^{p}{}_{r}\,\Gamma^{Mqr}\,\mathcal H^{CN}\,\Phi_{Npq}\,\partial_{C}\\&\quad{}+128\,\Gamma^{C}{}_{pq}\,\Gamma^{Mp}{}_{r}\,\Phi_{M}{}^{qr}\,\partial_{C}\end{aligned}$

...

${}\begin{aligned}&\quad{}-8\,\Gamma^{Mq}{}_{s}\,\mathcal H^{NP}\,\Phi_{M}{}^{rs}\,\Phi_{Npq}\,\Phi_{P}{}^{p}{}_{r}\\&\quad{}+2\,\Gamma^{M}{}_{rs}\,\mathcal H^{NP}\,\Phi_{M}{}^{rs}\,\Phi_{Npq}\,\Phi_{P}{}^{pq}\\&\quad{}+8\,\Gamma^{Mr}{}_{s}\,\mathcal H^{NP}\,\Phi_{M}{}^{qs}\,\Phi_{Npq}\,\Phi_{P}{}^{p}{}_{r}\\&\quad{}+2\,\Gamma^{Mrs}\,\mathcal H^{NP}\,\Phi_{M}{}^{pq}\,\Phi_{Npq}\,\Phi_{Prs}\\&\quad{}-16\,\partial_{A}\partial_{B}\Gamma_{M\bar p\bar q}\,\Gamma^{M\bar p\bar q}\,\mathcal H^{AB}\\&\quad{}-16\,\partial_{A}\Gamma_{M\bar p\bar q}\,\partial_{B}\Gamma^{M\bar p\bar q}\,\mathcal H^{AB}\\&\quad{}-16\,\partial_{B}\Gamma_{M\bar p\bar q}\,\partial_{A}\Gamma^{M\bar p\bar q}\,\mathcal H^{AB}\\&\quad{}-4\,\Gamma_{M\bar p\bar q}\,\Gamma_{N}{}^{\bar p\bar q}\,\Gamma^{M}{}_{\bar r\bar s}\,\Gamma^{N\bar r\bar s}\\&\quad{}+8\,\Gamma_{M\bar p\bar q}\,\Gamma_{N}{}^{\bar p\bar r}\,\Gamma^{M}{}_{\bar r\bar s}\,\Gamma^{N\bar q\bar s}\\&\quad{}+8\,\Gamma_{M\bar p\bar q}\,\Gamma_{N}{}^{\bar q}{}_{\bar s}\,\Gamma^{M\bar p}{}_{\bar r}\,\Gamma^{N\bar r\bar s}\\&\quad{}-4\,\Gamma_{M\bar p\bar q}\,\Gamma_{N\bar r\bar s}\,\Gamma^{M\bar p\bar q}\,\Gamma^{N\bar r\bar s}\\&\quad{}-8\,\Gamma_{M\bar p\bar q}\,\Gamma_{N}{}^{\bar r}{}_{\bar s}\,\Gamma^{M\bar p}{}_{\bar r}\,\Gamma^{N\bar q\bar s}\\&\quad{}-4\,\Gamma_{M\bar p\bar q}\,\Gamma_{N}{}^{\bar r\bar s}\,\Gamma^{M}{}_{\bar r\bar s}\,\Gamma^{N\bar p\bar q}\\&\quad{}-16\,\Gamma_{M\bar p\bar q}\,\partial_{A}\partial_{B}\Gamma^{M\bar p\bar q}\,\mathcal H^{AB}\\&\quad{}+4\,\Gamma_{M\bar p\bar q}\,\Gamma^{M\bar p\bar q}\,\mathcal H^{NP}\,\Phi_{Npq}\,\Phi_{P}{}^{pq}\end{aligned}$

${}\begin{aligned}&\quad{}-32\,\Gamma_{M\bar p\bar q}\,\Gamma^{M\bar p}{}_{\bar r}\,\mathfrak R^{[\bar q\bar r]}\\&\quad{}+32\,\partial_{E}\Gamma_{M}{}^{\bar p}{}_{\bar r}\,\Gamma^{E}{}_{\bar p\bar q}\,\Gamma^{M\bar q\bar r}\\&\quad{}+32\,\Gamma_{M}{}^{\bar p}{}_{\bar r}\,\Gamma^{E}{}_{\bar p\bar q}\,\partial_{E}\Gamma^{M\bar q\bar r}\\&\quad{}-32\,\Gamma_{M}{}^{\bar p}{}_{\bar r}\,\Gamma^{M\bar q\bar r}\,\mathfrak R_{[\bar p\bar q]}\\&\quad{}-64\,\Gamma^{E}{}_{\bar p\bar q}\,\partial_{E}\mathfrak R^{[\bar p\bar q]}\\&\quad{}-8\,\Gamma^{M}{}_{\bar p\bar q}\,\Gamma^{N\bar p\bar q}\,\Phi_{Mpq}\,\Phi_{N}{}^{pq}\\&\quad{}+8\,\Gamma^{M}{}_{\bar p\bar q}\,\mathfrak R^{pq\bar p\bar q}\,\Phi_{Mpq}\\&\quad{}-8\,\Gamma^{M}{}_{\bar p\bar q}\,\mathfrak R^{\bar p\bar qpq}\,\Phi_{Mpq}\\&\quad{}+8\,\Gamma^{M\bar p\bar q}\,\mathfrak R_{pq\bar p\bar q}\,\Phi_{M}{}^{pq}\\&\quad{}-8\,\Gamma^{M\bar p\bar q}\,\mathfrak R_{\bar p\bar qpq}\,\Phi_{M}{}^{pq}\\&\quad{}+8\,\mathcal H^{AB}\,\partial_{A}\partial_{B}\mathcal H^{MN}\,\Phi_{Mpq}\,\Phi_{N}{}^{pq}\\&\quad{}+8\,\mathcal H^{AB}\,\partial_{A}\mathcal H^{MN}\,\partial_{B}\Phi_{Mpq}\,\Phi_{N}{}^{pq}\\&\quad{}+8\,\mathcal H^{AB}\,\partial_{A}\mathcal H^{MN}\,\Phi_{Mpq}\,\partial_{B}\Phi_{N}{}^{pq}\\&\quad{}+8\,\mathcal H^{AB}\,\partial_{B}\mathcal H^{MN}\,\partial_{A}\Phi_{Mpq}\,\Phi_{N}{}^{pq}\\&\quad{}+8\,\mathcal H^{AB}\,\partial_{B}\mathcal H^{MN}\,\Phi_{Mpq}\,\partial_{A}\Phi_{N}{}^{pq}\end{aligned}$

${}\begin{aligned}&\quad{}+8\,\mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{A}\partial_{B}\Phi_{Mpq}\,\Phi_{N}{}^{pq}\\&\quad{}+8\,\mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{A}\Phi_{Mpq}\,\partial_{B}\Phi_{N}{}^{pq}\\&\quad{}+8\,\mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{B}\Phi_{Mpq}\,\partial_{A}\Phi_{N}{}^{pq}\\&\quad{}+8\,\mathcal H^{AB}\,\mathcal H^{MN}\,\Phi_{Mpq}\,\partial_{A}\partial_{B}\Phi_{N}{}^{pq}\\&\quad{}-16\,\mathcal H^{EM}\,\partial_{E}\mathcal H^{NP}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\Phi_{P}{}^{qr}\\&\quad{}+16\,\mathcal H^{EM}\,\partial_{E}\mathcal H^{NP}\,\Phi_{Mpq}\,\partial_{N}\Phi_{P}{}^{pq}\\&\quad{}-16\,\mathcal H^{EM}\,\mathcal H^{NP}\,\Phi_{Mpq}\,\partial_{E}\Phi_{N}{}^{p}{}_{r}\,\Phi_{P}{}^{qr}\\&\quad{}-16\,\mathcal H^{EM}\,\mathcal H^{NP}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\partial_{E}\Phi_{P}{}^{qr}\\&\quad{}+16\,\mathcal H^{EM}\,\mathcal H^{NP}\,\Phi_{Mpq}\,\partial_{E}\partial_{N}\Phi_{P}{}^{pq}\\&\quad{}+32\,\mathcal H^{EM}\,\Phi_{Mpq}\,\partial_{E}\mathfrak R^{[pq]}\\&\quad{}-\mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{N}{}^{pq}\,\Phi_{Prs}\,\Phi_{Q}{}^{rs}\\&\quad{}+4\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\Phi_{P}{}^{q}{}_{s}\,\Phi_{Q}{}^{rs}\\&\quad{}-4\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\Phi_{P}{}^{r}{}_{s}\,\Phi_{Q}{}^{qs}\\&\quad{}-8\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\partial_{P}\Phi_{Q}{}^{qr}\\&\quad{}-\mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{Nrs}\,\Phi_{P}{}^{pq}\,\Phi_{Q}{}^{rs}\end{aligned}$

${}\begin{aligned}&\quad{}+4\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{Nrs}\,\Phi_{P}{}^{pr}\,\Phi_{Q}{}^{qs}\\&\quad{}-\mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{Nrs}\,\Phi_{P}{}^{rs}\,\Phi_{Q}{}^{pq}\\&\quad{}-8\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\partial_{M}\Phi_{Npq}\,\Phi_{P}{}^{p}{}_{r}\,\Phi_{Q}{}^{qr}\\&\quad{}+8\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\partial_{M}\Phi_{Npq}\,\partial_{P}\Phi_{Q}{}^{pq}\\&\quad{}-16\,\mathcal H^{MN}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\mathfrak R^{[qr]}\\&\quad{}-16\,\mathcal H^{MN}\,\Phi_{M}{}^{p}{}_{r}\,\Phi_{N}{}^{qr}\,\mathfrak R_{[pq]}\\&\quad{}+16\,\mathcal H^{MN}\,\partial_{M}\Phi_{Npq}\,\mathfrak R^{[pq]}\\&\quad{}+16\,\mathcal H^{MN}\,\partial_{M}\Phi_{N}{}^{pq}\,\mathfrak R_{[pq]}\\&\quad{}-8\,\mathfrak R_{pq\bar p\bar q}\,\mathfrak R^{pq\bar p\bar q}\\&\quad{}+8\,\mathfrak R_{pq\bar p\bar q}\,\mathfrak R^{\bar p\bar qpq}\\&\quad{}+8\,\mathfrak R^{pq\bar p\bar q}\,\mathfrak R_{\bar p\bar qpq}\\&\quad{}-8\,\mathfrak R_{\bar p\bar qpq}\,\mathfrak R^{\bar p\bar qpq}\\&\quad{}+32\,\mathfrak R_{[pq]}\,\mathfrak R^{[pq]}\\&\quad{}+64\,\mathfrak R_{[\bar p\bar q]}\,\mathfrak R^{[\bar p\bar q]}\end{aligned}$

Prepared 8 weighted Cadabra variables with 1994 fully expanded terms.
F1: field=T, weight=1, terms=404
F2: field=phi, weight=128, terms=14
F3: field=BLL, weight=(1/4), terms=192
F4: field=BRR, weight=(1/4), terms=192
F5: field=UL, weight=-12, terms=192
F6: field=UR, weight=-12, terms=192
F7: field=ULLR, weight=(-1/64), terms=404
F8: field=ULRR, weight=(-1/64), terms=404
stored variables: F1, F2, F3, F4, F5, F6, F7, F8
Cadabra payload: actual tensor/partial expressions; K-labels: 0
all expanded rows: 1994


## 5. `F1 + F2 + ... + F8`을 직접 실행

이 셀에서는 아직 `collect_terms`를 호출하지 않습니다. 따라서 기본 $n=2$ 조합에서는 소거 전 1,994개 summand가 `totalTr` 안에 실제로 남아 있어야 합니다.

In [7]:
totalTr = F1 + F2 + F3 + F4 + F5 + F6 + F7 + F8

terms_before_collection = cadabra_term_count(totalTr)
assert terms_before_collection == PREPARED.term_count
print('executed: totalTr = F1 + F2 + F3 + F4 + F5 + F6 + F7 + F8')
print('summands before collect_terms:', terms_before_collection)

executed: totalTr = F1 + F2 + F3 + F4 + F5 + F6 + F7 + F8
summands before collect_terms: 1994


## 6. Cadabra로 직접 모아서 `totalTr = 0` 확인

먼저 404개 canonical tensor body 각각에 대한 weighted coefficient 합과 그 body의 전체 Einstein-contracted 수식을 출력합니다. 그 다음 바로 위 셀에서 만든 `totalTr` 자체에 `collect_terms`를 적용합니다.

In [8]:
_display_latex_chunks(_ledger_latex_chunks(PREPARED.ledger))

distribute(totalTr)


${}\begin{aligned}K0001&:\quad \left(256\right)_{T}+\left(128\right)_{\phi}+\left(64\right)_{B_{LL}}+\left(64\right)_{B_{RR}}+\left(-192\right)_{U_L}+\left(-192\right)_{U_R}+\left(-64\right)_{U_{LLR}}+\left(-64\right)_{U_{LRR}}=0,\qquad K0001\equiv \mathcal H^{AB}\,\mathcal H^{CD}\,\partial_{A}\,\partial_{B}\,\partial_{C}\,\partial_{D}\\K0002&:\quad \left(512\right)_{T}+\left(256\right)_{\phi}+\left(128\right)_{B_{LL}}+\left(128\right)_{B_{RR}}+\left(-384\right)_{U_L}+\left(-384\right)_{U_R}+\left(-128\right)_{U_{LLR}}+\left(-128\right)_{U_{LRR}}=0,\qquad K0002\equiv \Gamma_{MN}{}^{C}\,\mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{A}\,\partial_{B}\,\partial_{C}\\K0003&:\quad \left(512\right)_{T}+\left(256\right)_{\phi}+\left(128\right)_{B_{LL}}+\left(128\right)_{B_{RR}}+\left(-384\right)_{U_L}+\left(-384\right)_{U_R}+\left(-128\right)_{U_{LLR}}+\left(-128\right)_{U_{LRR}}=0,\qquad K0003\equiv \mathcal H^{AB}\,\partial_{A}\mathcal H^{CD}\,\partial_{B}\,\partial_{C}\,\partial_{D}\\K0004&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{RR}}+\left(48\right)_{U_R}+\left(16\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0004\equiv \bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{A}\,\partial_{B}\\K0005&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0005\equiv \bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{CN}\,\mathcal H^{EM}\,\partial_{E}\,\partial_{C}\\K0006&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0006\equiv \bar\Phi_{M\bar p\bar q}\,\Gamma^{C\bar p\bar q}\,\mathcal H^{EM}\,\partial_{E}\,\partial_{C}\\K0007&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0007\equiv \bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma^{E}{}_{\bar p\bar q}\,\mathcal H^{CM}\,\partial_{E}\,\partial_{C}\\K0008&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0008\equiv \bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma^{M}{}_{\bar p\bar q}\,\mathcal H^{AB}\,\partial_{A}\,\partial_{B}\end{aligned}$

${}\begin{aligned}K0009&:\quad \left(512\right)_{T}+\left(256\right)_{\phi}+\left(128\right)_{B_{LL}}+\left(128\right)_{B_{RR}}+\left(-384\right)_{U_L}+\left(-384\right)_{U_R}+\left(-128\right)_{U_{LLR}}+\left(-128\right)_{U_{LRR}}=0,\qquad K0009\equiv \partial_{A}\Gamma_{MN}{}^{C}\,\mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{B}\,\partial_{C}\\K0010&:\quad \left(512\right)_{T}+\left(256\right)_{\phi}+\left(128\right)_{B_{LL}}+\left(128\right)_{B_{RR}}+\left(-384\right)_{U_L}+\left(-384\right)_{U_R}+\left(-128\right)_{U_{LLR}}+\left(-128\right)_{U_{LRR}}=0,\qquad K0010\equiv \Gamma_{MN}{}^{C}\,\mathcal H^{AB}\,\partial_{A}\mathcal H^{MN}\,\partial_{B}\,\partial_{C}\\K0011&:\quad \left(256\right)_{T}+\left(128\right)_{\phi}+\left(64\right)_{B_{LL}}+\left(64\right)_{B_{RR}}+\left(-192\right)_{U_L}+\left(-192\right)_{U_R}+\left(-64\right)_{U_{LLR}}+\left(-64\right)_{U_{LRR}}=0,\qquad K0011\equiv \Gamma_{MN}{}^{E}\,\Gamma_{PQ}{}^{C}\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\partial_{E}\,\partial_{C}\\K0012&:\quad \left(256\right)_{T}+\left(128\right)_{\phi}+\left(64\right)_{B_{LL}}+\left(64\right)_{B_{RR}}+\left(-192\right)_{U_L}+\left(-192\right)_{U_R}+\left(-64\right)_{U_{LLR}}+\left(-64\right)_{U_{LRR}}=0,\qquad K0012\equiv \Gamma_{MN}{}^{E}\,\partial_{E}\mathcal H^{CD}\,\mathcal H^{MN}\,\partial_{C}\,\partial_{D}\\K0013&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{LL}}+\left(48\right)_{U_L}+\left(32\right)_{U_{LLR}}+\left(16\right)_{U_{LRR}}=0,\qquad K0013\equiv \Gamma_{Mpq}\,\Gamma^{Mpq}\,\mathcal H^{AB}\,\partial_{A}\,\partial_{B}\\K0014&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{LL}}+\left(96\right)_{U_L}+\left(64\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0014\equiv \Gamma^{Cpq}\,\Gamma^{E}{}_{pq}\,\partial_{E}\,\partial_{C}\\K0015&:\quad \left(128\right)_{T}+\left(64\right)_{B_{LL}}+\left(-96\right)_{U_L}+\left(-64\right)_{U_{LLR}}+\left(-32\right)_{U_{LRR}}=0,\qquad K0015\equiv \Gamma^{Cpq}\,\mathcal H^{EM}\,\Phi_{Mpq}\,\partial_{E}\,\partial_{C}\\K0016&:\quad \left(128\right)_{T}+\left(64\right)_{B_{LL}}+\left(-96\right)_{U_L}+\left(-64\right)_{U_{LLR}}+\left(-32\right)_{U_{LRR}}=0,\qquad K0016\equiv \Gamma^{E}{}_{pq}\,\mathcal H^{CM}\,\Phi_{M}{}^{pq}\,\partial_{E}\,\partial_{C}\end{aligned}$

${}\begin{aligned}K0017&:\quad \left(128\right)_{T}+\left(64\right)_{B_{LL}}+\left(-96\right)_{U_L}+\left(-64\right)_{U_{LLR}}+\left(-32\right)_{U_{LRR}}=0,\qquad K0017\equiv \Gamma^{M}{}_{pq}\,\mathcal H^{AB}\,\Phi_{M}{}^{pq}\,\partial_{A}\,\partial_{B}\\K0018&:\quad \left(64\right)_{T}+\left(32\right)_{B_{RR}}+\left(-48\right)_{U_R}+\left(-16\right)_{U_{LLR}}+\left(-32\right)_{U_{LRR}}=0,\qquad K0018\equiv \Gamma_{M\bar p\bar q}\,\Gamma^{M\bar p\bar q}\,\mathcal H^{AB}\,\partial_{A}\,\partial_{B}\\K0019&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0019\equiv \Gamma^{C\bar p\bar q}\,\Gamma^{E}{}_{\bar p\bar q}\,\partial_{E}\,\partial_{C}\\K0020&:\quad \left(256\right)_{T}+\left(128\right)_{\phi}+\left(64\right)_{B_{LL}}+\left(64\right)_{B_{RR}}+\left(-192\right)_{U_L}+\left(-192\right)_{U_R}+\left(-64\right)_{U_{LLR}}+\left(-64\right)_{U_{LRR}}=0,\qquad K0020\equiv \mathcal H^{AB}\,\partial_{A}\partial_{B}\mathcal H^{CD}\,\partial_{C}\,\partial_{D}\\K0021&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{LL}}+\left(48\right)_{U_L}+\left(32\right)_{U_{LLR}}+\left(16\right)_{U_{LRR}}=0,\qquad K0021\equiv \mathcal H^{AB}\,\mathcal H^{MN}\,\Phi_{Mpq}\,\Phi_{N}{}^{pq}\,\partial_{A}\,\partial_{B}\\K0022&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{LL}}+\left(96\right)_{U_L}+\left(64\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0022\equiv \mathcal H^{CM}\,\mathcal H^{EN}\,\Phi_{M}{}^{pq}\,\Phi_{Npq}\,\partial_{E}\,\partial_{C}\\K0023&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{RR}}+\left(48\right)_{U_R}+\left(16\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0023\equiv \partial_{A}\bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{AC}\,\mathcal H^{MN}\,\partial_{C}\\K0024&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{RR}}+\left(48\right)_{U_R}+\left(16\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0024\equiv \partial_{N}\bar\Phi_{M\bar p\bar q}\,\Gamma^{C\bar p\bar q}\,\mathcal H^{NM}\,\partial_{C}\end{aligned}$

${}\begin{aligned}K0025&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{RR}}+\left(48\right)_{U_R}+\left(16\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0025\equiv \bar\Phi_{M\bar p\bar q}\,\partial_{A}\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{AC}\,\mathcal H^{MN}\,\partial_{C}\\K0026&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0026\equiv \bar\Phi_{M\bar p\bar q}\,\partial_{E}\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{CN}\,\mathcal H^{EM}\,\partial_{C}\\K0027&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{RR}}+\left(48\right)_{U_R}+\left(16\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0027\equiv \bar\Phi_{M\bar p\bar q}\,\partial_{P}\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{CM}\,\mathcal H^{PN}\,\partial_{C}\\K0028&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{RR}}+\left(48\right)_{U_R}+\left(16\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0028\equiv \bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\Gamma_{PQ}{}^{C}\,\mathcal H^{MN}\,\mathcal H^{PQ}\,\partial_{C}\\K0029&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{RR}}+\left(48\right)_{U_R}+\left(16\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0029\equiv \bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\Gamma_{PQ}{}^{M}\,\mathcal H^{CN}\,\mathcal H^{PQ}\,\partial_{C}\\K0030&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{RR}}+\left(48\right)_{U_R}+\left(16\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0030\equiv \bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\mathcal H^{AC}\,\partial_{A}\mathcal H^{MN}\,\partial_{C}\\K0031&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0031\equiv \bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p\bar q}\,\partial_{E}\mathcal H^{CN}\,\mathcal H^{EM}\,\partial_{C}\\K0032&:\quad \left(64\right)_{T}+\left(32\right)_{B_{RR}}+\left(-48\right)_{U_R}+\left(-16\right)_{U_{LLR}}+\left(-32\right)_{U_{LRR}}=0,\qquad K0032\equiv \bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p}{}_{\bar r}\,\bar\Phi_{P}{}^{\bar q\bar r}\,\mathcal H^{CM}\,\mathcal H^{NP}\,\partial_{C}\end{aligned}$

${}\begin{aligned}K0033&:\quad \left(64\right)_{T}+\left(32\right)_{B_{RR}}+\left(-48\right)_{U_R}+\left(-16\right)_{U_{LLR}}+\left(-32\right)_{U_{LRR}}=0,\qquad K0033\equiv \bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar p}{}_{\bar r}\,\Gamma^{C\bar q\bar r}\,\mathcal H^{MN}\,\partial_{C}\\K0034&:\quad \left(64\right)_{T}+\left(32\right)_{B_{RR}}+\left(-48\right)_{U_R}+\left(-16\right)_{U_{LLR}}+\left(-32\right)_{U_{LRR}}=0,\qquad K0034\equiv \bar\Phi_{M\bar p\bar q}\,\bar\Phi_{N}{}^{\bar q\bar r}\,\bar\Phi_{P}{}^{\bar p}{}_{\bar r}\,\mathcal H^{CN}\,\mathcal H^{MP}\,\partial_{C}\\K0035&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{RR}}+\left(48\right)_{U_R}+\left(16\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0035\equiv \bar\Phi_{M\bar p\bar q}\,\Gamma_{NP}{}^{M}\,\Gamma^{C\bar p\bar q}\,\mathcal H^{NP}\,\partial_{C}\\K0036&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{RR}}+\left(48\right)_{U_R}+\left(16\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0036\equiv \bar\Phi_{M\bar p\bar q}\,\Gamma_{N}{}^{\bar p}{}_{\bar r}\,\Gamma^{N\bar q\bar r}\,\mathcal H^{CM}\,\partial_{C}\\K0037&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0037\equiv \bar\Phi_{M\bar p\bar q}\,\partial_{E}\Gamma^{C\bar p\bar q}\,\mathcal H^{EM}\,\partial_{C}\\K0038&:\quad \left(128\right)_{T}+\left(64\right)_{B_{RR}}+\left(-96\right)_{U_R}+\left(-32\right)_{U_{LLR}}+\left(-64\right)_{U_{LRR}}=0,\qquad K0038\equiv \bar\Phi_{M\bar p\bar q}\,\mathcal H^{CM}\,\mathfrak R^{[\bar p\bar q]}\,\partial_{C}\\K0039&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0039\equiv \partial_{A}\bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma^{M}{}_{\bar p\bar q}\,\mathcal H^{AC}\,\partial_{C}\\K0040&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0040\equiv \partial_{E}\bar\Phi_{M}{}^{\bar p\bar q}\,\Gamma^{E}{}_{\bar p\bar q}\,\mathcal H^{CM}\,\partial_{C}\end{aligned}$

...

${}\begin{aligned}K0369&:\quad \left(-16\right)_{T}+\left(8\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0369\equiv \Gamma^{M\bar p\bar q}\,\mathfrak R_{pq\bar p\bar q}\,\Phi_{M}{}^{pq}\\K0370&:\quad \left(16\right)_{T}+\left(-8\right)_{U_{LLR}}+\left(-8\right)_{U_{LRR}}=0,\qquad K0370\equiv \Gamma^{M\bar p\bar q}\,\mathfrak R_{\bar p\bar qpq}\,\Phi_{M}{}^{pq}\\K0371&:\quad \left(-32\right)_{T}+\left(-16\right)_{B_{LL}}+\left(24\right)_{U_L}+\left(16\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0371\equiv \mathcal H^{AB}\,\partial_{A}\partial_{B}\mathcal H^{MN}\,\Phi_{Mpq}\,\Phi_{N}{}^{pq}\\K0372&:\quad \left(-32\right)_{T}+\left(-16\right)_{B_{LL}}+\left(24\right)_{U_L}+\left(16\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0372\equiv \mathcal H^{AB}\,\partial_{A}\mathcal H^{MN}\,\partial_{B}\Phi_{Mpq}\,\Phi_{N}{}^{pq}\\K0373&:\quad \left(-32\right)_{T}+\left(-16\right)_{B_{LL}}+\left(24\right)_{U_L}+\left(16\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0373\equiv \mathcal H^{AB}\,\partial_{A}\mathcal H^{MN}\,\Phi_{Mpq}\,\partial_{B}\Phi_{N}{}^{pq}\\K0374&:\quad \left(-32\right)_{T}+\left(-16\right)_{B_{LL}}+\left(24\right)_{U_L}+\left(16\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0374\equiv \mathcal H^{AB}\,\partial_{B}\mathcal H^{MN}\,\partial_{A}\Phi_{Mpq}\,\Phi_{N}{}^{pq}\\K0375&:\quad \left(-32\right)_{T}+\left(-16\right)_{B_{LL}}+\left(24\right)_{U_L}+\left(16\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0375\equiv \mathcal H^{AB}\,\partial_{B}\mathcal H^{MN}\,\Phi_{Mpq}\,\partial_{A}\Phi_{N}{}^{pq}\\K0376&:\quad \left(-32\right)_{T}+\left(-16\right)_{B_{LL}}+\left(24\right)_{U_L}+\left(16\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0376\equiv \mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{A}\partial_{B}\Phi_{Mpq}\,\Phi_{N}{}^{pq}\end{aligned}$

${}\begin{aligned}K0377&:\quad \left(-32\right)_{T}+\left(-16\right)_{B_{LL}}+\left(24\right)_{U_L}+\left(16\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0377\equiv \mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{A}\Phi_{Mpq}\,\partial_{B}\Phi_{N}{}^{pq}\\K0378&:\quad \left(-32\right)_{T}+\left(-16\right)_{B_{LL}}+\left(24\right)_{U_L}+\left(16\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0378\equiv \mathcal H^{AB}\,\mathcal H^{MN}\,\partial_{B}\Phi_{Mpq}\,\partial_{A}\Phi_{N}{}^{pq}\\K0379&:\quad \left(-32\right)_{T}+\left(-16\right)_{B_{LL}}+\left(24\right)_{U_L}+\left(16\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0379\equiv \mathcal H^{AB}\,\mathcal H^{MN}\,\Phi_{Mpq}\,\partial_{A}\partial_{B}\Phi_{N}{}^{pq}\\K0380&:\quad \left(64\right)_{T}+\left(32\right)_{B_{LL}}+\left(-48\right)_{U_L}+\left(-32\right)_{U_{LLR}}+\left(-16\right)_{U_{LRR}}=0,\qquad K0380\equiv \mathcal H^{EM}\,\partial_{E}\mathcal H^{NP}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\Phi_{P}{}^{qr}\\K0381&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{LL}}+\left(48\right)_{U_L}+\left(32\right)_{U_{LLR}}+\left(16\right)_{U_{LRR}}=0,\qquad K0381\equiv \mathcal H^{EM}\,\partial_{E}\mathcal H^{NP}\,\Phi_{Mpq}\,\partial_{N}\Phi_{P}{}^{pq}\\K0382&:\quad \left(64\right)_{T}+\left(32\right)_{B_{LL}}+\left(-48\right)_{U_L}+\left(-32\right)_{U_{LLR}}+\left(-16\right)_{U_{LRR}}=0,\qquad K0382\equiv \mathcal H^{EM}\,\mathcal H^{NP}\,\Phi_{Mpq}\,\partial_{E}\Phi_{N}{}^{p}{}_{r}\,\Phi_{P}{}^{qr}\\K0383&:\quad \left(64\right)_{T}+\left(32\right)_{B_{LL}}+\left(-48\right)_{U_L}+\left(-32\right)_{U_{LLR}}+\left(-16\right)_{U_{LRR}}=0,\qquad K0383\equiv \mathcal H^{EM}\,\mathcal H^{NP}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\partial_{E}\Phi_{P}{}^{qr}\\K0384&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{LL}}+\left(48\right)_{U_L}+\left(32\right)_{U_{LLR}}+\left(16\right)_{U_{LRR}}=0,\qquad K0384\equiv \mathcal H^{EM}\,\mathcal H^{NP}\,\Phi_{Mpq}\,\partial_{E}\partial_{N}\Phi_{P}{}^{pq}\end{aligned}$

${}\begin{aligned}K0385&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{LL}}+\left(96\right)_{U_L}+\left(64\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0385\equiv \mathcal H^{EM}\,\Phi_{Mpq}\,\partial_{E}\mathfrak R^{[pq]}\\K0386&:\quad \left(4\right)_{T}+\left(4\right)_{B_{LL}}+\left(-3\right)_{U_L}+\left(-4\right)_{U_{LLR}}+\left(-1\right)_{U_{LRR}}=0,\qquad K0386\equiv \mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{N}{}^{pq}\,\Phi_{Prs}\,\Phi_{Q}{}^{rs}\\K0387&:\quad \left(-16\right)_{T}+\left(-8\right)_{B_{LL}}+\left(12\right)_{U_L}+\left(8\right)_{U_{LLR}}+\left(4\right)_{U_{LRR}}=0,\qquad K0387\equiv \mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\Phi_{P}{}^{q}{}_{s}\,\Phi_{Q}{}^{rs}\\K0388&:\quad \left(16\right)_{T}+\left(8\right)_{B_{LL}}+\left(-12\right)_{U_L}+\left(-8\right)_{U_{LLR}}+\left(-4\right)_{U_{LRR}}=0,\qquad K0388\equiv \mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\Phi_{P}{}^{r}{}_{s}\,\Phi_{Q}{}^{qs}\\K0389&:\quad \left(32\right)_{T}+\left(16\right)_{B_{LL}}+\left(-24\right)_{U_L}+\left(-16\right)_{U_{LLR}}+\left(-8\right)_{U_{LRR}}=0,\qquad K0389\equiv \mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\partial_{P}\Phi_{Q}{}^{qr}\\K0390&:\quad \left(4\right)_{T}+\left(4\right)_{B_{LL}}+\left(-3\right)_{U_L}+\left(-4\right)_{U_{LLR}}+\left(-1\right)_{U_{LRR}}=0,\qquad K0390\equiv \mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{Nrs}\,\Phi_{P}{}^{pq}\,\Phi_{Q}{}^{rs}\\K0391&:\quad \left(-16\right)_{T}+\left(-8\right)_{B_{LL}}+\left(12\right)_{U_L}+\left(8\right)_{U_{LLR}}+\left(4\right)_{U_{LRR}}=0,\qquad K0391\equiv \mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{Nrs}\,\Phi_{P}{}^{pr}\,\Phi_{Q}{}^{qs}\\K0392&:\quad \left(4\right)_{T}+\left(4\right)_{B_{LL}}+\left(-3\right)_{U_L}+\left(-4\right)_{U_{LLR}}+\left(-1\right)_{U_{LRR}}=0,\qquad K0392\equiv \mathcal H^{MN}\,\mathcal H^{PQ}\,\Phi_{Mpq}\,\Phi_{Nrs}\,\Phi_{P}{}^{rs}\,\Phi_{Q}{}^{pq}\end{aligned}$

${}\begin{aligned}K0393&:\quad \left(32\right)_{T}+\left(16\right)_{B_{LL}}+\left(-24\right)_{U_L}+\left(-16\right)_{U_{LLR}}+\left(-8\right)_{U_{LRR}}=0,\qquad K0393\equiv \mathcal H^{MN}\,\mathcal H^{PQ}\,\partial_{M}\Phi_{Npq}\,\Phi_{P}{}^{p}{}_{r}\,\Phi_{Q}{}^{qr}\\K0394&:\quad \left(-32\right)_{T}+\left(-16\right)_{B_{LL}}+\left(24\right)_{U_L}+\left(16\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0394\equiv \mathcal H^{MN}\,\mathcal H^{PQ}\,\partial_{M}\Phi_{Npq}\,\partial_{P}\Phi_{Q}{}^{pq}\\K0395&:\quad \left(64\right)_{T}+\left(32\right)_{B_{LL}}+\left(-48\right)_{U_L}+\left(-32\right)_{U_{LLR}}+\left(-16\right)_{U_{LRR}}=0,\qquad K0395\equiv \mathcal H^{MN}\,\Phi_{Mpq}\,\Phi_{N}{}^{p}{}_{r}\,\mathfrak R^{[qr]}\\K0396&:\quad \left(64\right)_{T}+\left(32\right)_{B_{LL}}+\left(-48\right)_{U_L}+\left(-32\right)_{U_{LLR}}+\left(-16\right)_{U_{LRR}}=0,\qquad K0396\equiv \mathcal H^{MN}\,\Phi_{M}{}^{p}{}_{r}\,\Phi_{N}{}^{qr}\,\mathfrak R_{[pq]}\\K0397&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{LL}}+\left(48\right)_{U_L}+\left(32\right)_{U_{LLR}}+\left(16\right)_{U_{LRR}}=0,\qquad K0397\equiv \mathcal H^{MN}\,\partial_{M}\Phi_{Npq}\,\mathfrak R^{[pq]}\\K0398&:\quad \left(-64\right)_{T}+\left(-32\right)_{B_{LL}}+\left(48\right)_{U_L}+\left(32\right)_{U_{LLR}}+\left(16\right)_{U_{LRR}}=0,\qquad K0398\equiv \mathcal H^{MN}\,\partial_{M}\Phi_{N}{}^{pq}\,\mathfrak R_{[pq]}\\K0399&:\quad \left(16\right)_{T}+\left(-8\right)_{U_{LLR}}+\left(-8\right)_{U_{LRR}}=0,\qquad K0399\equiv \mathfrak R_{pq\bar p\bar q}\,\mathfrak R^{pq\bar p\bar q}\\K0400&:\quad \left(-16\right)_{T}+\left(8\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0400\equiv \mathfrak R_{pq\bar p\bar q}\,\mathfrak R^{\bar p\bar qpq}\end{aligned}$

${}\begin{aligned}K0401&:\quad \left(-16\right)_{T}+\left(8\right)_{U_{LLR}}+\left(8\right)_{U_{LRR}}=0,\qquad K0401\equiv \mathfrak R^{pq\bar p\bar q}\,\mathfrak R_{\bar p\bar qpq}\\K0402&:\quad \left(16\right)_{T}+\left(-8\right)_{U_{LLR}}+\left(-8\right)_{U_{LRR}}=0,\qquad K0402\equiv \mathfrak R_{\bar p\bar qpq}\,\mathfrak R^{\bar p\bar qpq}\\K0403&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{LL}}+\left(96\right)_{U_L}+\left(64\right)_{U_{LLR}}+\left(32\right)_{U_{LRR}}=0,\qquad K0403\equiv \mathfrak R_{[pq]}\,\mathfrak R^{[pq]}\\K0404&:\quad \left(-128\right)_{T}+\left(-64\right)_{B_{RR}}+\left(96\right)_{U_R}+\left(32\right)_{U_{LLR}}+\left(64\right)_{U_{LRR}}=0,\qquad K0404\equiv \mathfrak R_{[\bar p\bar q]}\,\mathfrak R^{[\bar p\bar q]}\end{aligned}$

In [9]:
len(totalTr);

1994

In [10]:
canonicalise(totalTr)

In [11]:

collect_terms(totalTr)


In [12]:

print('totalTr =', totalTr)
display(totalTr)
assert totalTr == 0
assert all(row.residual == 0 for row in PREPARED.ledger)
print(f'VERIFIED: {len(PREPARED.ledger)}/{len(PREPARED.ledger)} tensor bodies cancel exactly')

${}0\,$

totalTr = 0
VERIFIED: 404/404 tensor bodies cancel exactly


## 7. Box 정의 불일치 negative control

같은 수정 Box를 여덟 필드 모두에 공통 적용하면, 이 weight가 representation moment를 각각 소거하기 때문에 결과가 계속 0일 수 있습니다. 따라서 **한 필드만 다른 연산자를 쓰는 의도적인 불일치**로 입력 연결을 검사합니다. 아래에서는 $T$의 $\Delta$에 있는 실제 tensor body $\Gamma_{Bq_1q_2}\Gamma^B{}_{q_3q_4}$를 $\mathfrak R_{q_1q_2q_3q_4}$로 교체하고, 정상 계산과 같은 `prepare → F1+⋯+F8 → Cadabra collect` 경로를 실행합니다. `Riemann(pair1,pair2)`는 각 pair 내부의 반대칭만 사용하며 pair-exchange symmetry나 Bianchi identity는 가정하지 않습니다.


In [13]:
RIEMANN_SQUARE_BODY_INPUT = BoxTensorBody(
    name='Riemann replacement',
    monomials=(TensorBodyMonomial(Fraction(1), (
        TensorBodyFactor('Riemann', ('pair1', 'pair2')),
    )),),
)
MODIFIED_T_BOX = BOX_INPUT.with_body(
    'delta', 'generator_square', RIEMANN_SQUARE_BODY_INPUT
)
MODIFIED_FIELD_BOX_DEFINITIONS = dict(FIELD_BOX_DEFINITIONS)
MODIFIED_FIELD_BOX_DEFINITIONS['T'] = MODIFIED_T_BOX

assert MODIFIED_T_BOX.delta.generator_square_body != BOX_INPUT.delta.generator_square_body
print('normal T square body  :', BOX_INPUT.delta.generator_square_body.latex('delta'))
print('modified T square body:', MODIFIED_T_BOX.delta.generator_square_body.latex('delta'))
show_box_definition(box_definition=MODIFIED_T_BOX)


${}\begin{aligned}\Box T:&=\Delta T\\&\quad{}-\bar\Delta T.\end{aligned}\tag{2.6}$

${}\begin{aligned}\Delta T&=\mathcal D_q\mathcal D^qT\\&\quad{}+\mathfrak R_{[q_1q_2]}G^{q_1q_2}T\\&\quad{}-\Gamma^B{}_{q_1q_2}\mathcal D_BG^{q_1q_2}T\\&\quad{}+\frac{1}{4}\,\mathfrak R_{q_1q_2q_3q_4}G^{q_1q_2}G^{q_3q_4}T\\&\quad{}+\frac{1}{2}\,\mathfrak R_{\bar q_1\bar q_2q_3q_4}\bar G^{\bar q_1\bar q_2}G^{q_3q_4}T.\end{aligned}\tag{2.1}$

${}\begin{aligned}\bar\Delta T&=\mathcal D_{\bar q}\mathcal D^{\bar q}T\\&\quad{}+\mathfrak R_{[\bar q_1\bar q_2]}\bar G^{\bar q_1\bar q_2}T\\&\quad{}-\Gamma^B{}_{\bar q_1\bar q_2}\mathcal D_B\bar G^{\bar q_1\bar q_2}T\\&\quad{}+\frac{1}{4}\,\Gamma_{B\bar q_1\bar q_2}\Gamma^B{}_{\bar q_3\bar q_4}\bar G^{\bar q_1\bar q_2}\bar G^{\bar q_3\bar q_4}T\\&\quad{}+\frac{1}{2}\,\mathfrak R_{q_1q_2\bar q_3\bar q_4}G^{q_1q_2}\bar G^{\bar q_3\bar q_4}T.\end{aligned}\tag{2.2}$

normal T square body  : \Gamma_{Bq_1q_2}\Gamma^B{}_{q_3q_4}
modified T square body: \mathfrak R_{q_1q_2q_3q_4}


In [14]:
MODIFIED_PREPARED = prepare_field_variables(
    FIELD_COMBINATION,
    N,
    box_definition=BOX_INPUT,
    field_box_definitions=MODIFIED_FIELD_BOX_DEFINITIONS,
    show=False,
)
MF1, MF2, MF3, MF4, MF5, MF6, MF7, MF8 = MODIFIED_PREPARED.weighted_expressions
totalTr_modified = MF1 + MF2 + MF3 + MF4 + MF5 + MF6 + MF7 + MF8

distribute(totalTr_modified)
modified_precollection_terms = cadabra_term_count(totalTr_modified)
modified_ledger_residuals = sum(
    row.residual != 0 for row in MODIFIED_PREPARED.ledger
)
canonicalise(totalTr_modified)
collect_terms(totalTr_modified)
modified_residual_terms = cadabra_term_count(totalTr_modified)

print('totalTr_modified == 0:', totalTr_modified == 0)
print('pre-collection summands:', modified_precollection_terms)
print('nonzero coefficient rows:', modified_ledger_residuals)
print('Cadabra residual terms  :', modified_residual_terms)
assert totalTr_modified != 0
assert modified_ledger_residuals > 0
assert modified_residual_terms > 0
print('NEGATIVE CONTROL VERIFIED: the T-only Box mismatch is detected.')


totalTr_modified == 0: False
pre-collection summands: 1987
nonzero coefficient rows: 107
Cadabra residual terms  : 91
NEGATIVE CONTROL VERIFIED: the T-only Box mismatch is detected.


## 계산 범위

정상 결과는 $D=10$, $\dim S=\dim\bar S=16$, density weight 0, unprojected raw tensor products와 여덟 필드가 공유하는 공통 background/operator에 대한 finite-dimensional representation trace입니다. 마지막 negative control은 검증기의 민감도를 확인하기 위해 이 공통-operator 가정을 $T$에서만 의도적으로 깨뜨립니다. Functional/momentum trace, field-dependent mass, statistics, Pfaffian, gauge fixing 및 ghost factor는 별도입니다. 동일한 숫자 weight를 그대로 쓰면 algebraic $n=1$ representation trace도 상쇄되지만, 실제 one-loop determinant의 물리적 $n=1$ prefactor가 같다는 뜻은 아닙니다.